# Fake News Detection using FLAN-T5

## Notebook 4: Fine-Tuning FLAN-T5

### Course
ADS-509 – Applied Large Language Models for Data Science

### Team Project

## Objective

This notebook fine-tunes Google's FLAN-T5 model on the Fake and Real News Dataset.

The objective is to adapt the pre-trained language model for binary fake news classification and compare its performance against the zero-shot and few-shot prompting approaches evaluated previously.

The fine-tuned model produced in this notebook will be evaluated in Notebook 5.

In [73]:
import pandas as pd
import numpy as np

from datasets import Dataset

from sklearn.model_selection import train_test_split

from transformers import (
    AutoTokenizer,
    AutoModelForSeq2SeqLM,
    DataCollatorForSeq2Seq,
    Seq2SeqTrainer,
    Seq2SeqTrainingArguments
)

import torch

import warnings
warnings.filterwarnings("ignore")

In [75]:
device = "cuda" if torch.cuda.is_available() else "cpu"

print("Device:", device)

Device: cpu


In [77]:
import os

print(os.path.getsize("clean_news.csv"))

114221577


In [79]:
import pandas as pd

df_test = pd.read_csv("clean_news.csv", nrows=5)
print(df_test)

                                             content  label
0  WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...      0
1  Kenya opposition leader calls for calm in slum...      1
2  Egypt rejects U.S. decision to move its embass...      1
3  (AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...      0
4   Trump Rally Nearly Turns Into A Full-Blown Ra...      0


In [81]:
import pandas as pd

try:
    test = pd.read_csv("clean_news.csv", nrows=5)
    print(test)
    print("First 5 rows loaded successfully.")
except Exception as e:
    print(e)

                                             content  label
0  WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...      0
1  Kenya opposition leader calls for calm in slum...      1
2  Egypt rejects U.S. decision to move its embass...      1
3  (AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...      0
4   Trump Rally Nearly Turns Into A Full-Blown Ra...      0
First 5 rows loaded successfully.


In [82]:
try:
    test = pd.read_csv("clean_news.csv")
    print(test.shape)
except Exception as e:
    print(e)

(44689, 2)


In [85]:
print(os.path.getsize("clean_news.csv"))

114221577


In [87]:
pd.read_csv("clean_news.csv", nrows=5)

,content,label
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,0
1,Kenya opposition leader calls for calm in slum...,1
2,Egypt rejects U.S. decision to move its embass...,1
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,0
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,0


In [89]:
pd.read_csv("clean_news.csv").shape

(44689, 2)

In [91]:
import csv

with open("clean_news.csv", "r", encoding="utf-8") as f:
    reader = csv.reader(f)

    count = 0
    for row in reader:
        count += 1

print("Rows read:", count)

Rows read: 44690


In [93]:
import pandas as pd

df = pd.read_csv(
    "clean_news.csv",
    engine="python"
)

df.head()

,content,label
0,WOW! LEFTIST LIBRARIAN REJECTS Shipment Of Chi...,0
1,Kenya opposition leader calls for calm in slum...,1
2,Egypt rejects U.S. decision to move its embass...,1
3,(AUDIO)NATION OF ISLAM LEADER FARRAKHAN: “WE W...,0
4,Trump Rally Nearly Turns Into A Full-Blown Ra...,0


In [95]:
print("Dataset Shape:", df.shape)

print("\nMissing Values")

print(df.isnull().sum())

print("\nClass Distribution")

print(df["label"].value_counts())

Dataset Shape: (44689, 2)

Missing Values
content    0
label      0
dtype: int64

Class Distribution
label
0    23478
1    21211
Name: count, dtype: int64


### Observation

- The cleaned dataset contains **44,689 news articles** and **2 columns** (`content` and `label`).
- No missing values were found in either column, indicating successful preprocessing in Notebook 1.
- The class distribution is relatively balanced, with **23,478 fake news articles (label = 0)** and **21,211 real news articles (label = 1)**.
- Since the dataset is clean and only slightly imbalanced, it is suitable for fine-tuning the FLAN-T5 model.

In [98]:
train_df, eval_df = train_test_split(
    df,
    test_size=0.2,
    random_state=42,
    stratify=df["label"]
)

print("Training Samples:", len(train_df))
print("Validation Samples:", len(eval_df))

Training Samples: 35751
Validation Samples: 8938


### Observation

- The dataset was divided into **80% training** and **20% validation** sets.
- Stratified sampling was used to preserve the original class distribution in both datasets.
- The training dataset contains **35,751** samples, while the validation dataset contains **8,938** samples.
- This split allows the model to learn from a large training set while evaluating its generalization on unseen data.

In [101]:
train_ds = Dataset.from_pandas(train_df.reset_index(drop=True))

eval_ds = Dataset.from_pandas(eval_df.reset_index(drop=True))

In [103]:
MODEL_NAME = "google/flan-t5-base"

tok = AutoTokenizer.from_pretrained(MODEL_NAME)

flan = AutoModelForSeq2SeqLM.from_pretrained(MODEL_NAME)

flan.to(device)

print("Model Loaded Successfully!")

Loading weights:   0%|          | 0/282 [00:00<?, ?it/s]

[transformers] The tied weights mapping and config for this model specifies to tie shared.weight to lm_head.weight, but both are present in the checkpoints with different values, so we will NOT tie them. You should update the config with `tie_word_embeddings=False` to silence this warning.


Model Loaded Successfully!


### Observation

- The FLAN-T5 Base model and tokenizer were successfully downloaded and loaded from the Hugging Face model repository.
- The model is now ready for prompt formatting, tokenization, and supervised fine-tuning.
- Although the notebook is currently running on the CPU, the model can still be fine-tuned successfully (training will take longer than on a GPU).
- This completes the environment setup required for model training.

In [106]:
def label_to_text(label):
    if label == 0:
        return "Fake"
    else:
        return "Real"

In [108]:
def format_data(example):

    input_text = (
        "Classify the following news article as Fake or Real.\n\n"
        f"News:\n{example['content']}\n\n"
        "Answer:"
    )

    target_text = label_to_text(example["label"])

    return {
        "input_text": input_text,
        "target_text": target_text
    }

In [110]:
train_formatted = train_ds.map(format_data)

eval_formatted = eval_ds.map(format_data)

Map:   0%|          | 0/35751 [00:00<?, ? examples/s]

Map:   0%|          | 0/8938 [00:00<?, ? examples/s]

In [112]:
print(train_formatted[0])

{'content': ' This Touching New Bernie Sanders Ad Will Leave You in Tears (VIDEO). Bernie Sanders candidacy is attractive to many different demographics. He s extremely appealing to younger generations who are the future of tomorrow with his ideas on education and income equality. He s also appealing on social justice issues like incarceration of African Americans and the plight of the Palestinian people, who have been under a brutal Israeli military occupation for decades. There is one other demographic that is backing Bernie Sanders for president: the American men and women who serve in the military.Bernie Sanders is the only candidate who voted against the Iraq war, correctly predicting its terrible ramifications. Millions of Iraqis have been killed, a once thriving economy decimated, and a power vacuum was created that lead to the rise of organizations like ISIS.As for the United States, trillions of dollars have been lost which lead to a recession under the Bush administration uns

### Observation

- The original numeric labels were converted into textual labels ("Fake" and "Real") to match the text generation nature of the FLAN-T5 model.
- Each news article was reformatted into an instruction-based prompt that asks the model to classify the article as Fake or Real.
- This instruction-following format aligns with FLAN-T5's pre-training objective and helps the model learn the classification task more effectively during fine-tuning.
- The formatted dataset now includes two additional fields, input_text (instruction + news article) and target_text (expected answer), while preserving the original content and label columns for now.

In [115]:
def tokenize_fn(batch):

    # Tokenize the input prompt
    model_inputs = tok(
        batch["input_text"],
        truncation=True,
        padding="max_length",
        max_length=512
    )

    # Tokenize the expected answer
    labels = tok(
        batch["target_text"],
        truncation=True,
        padding="max_length",
        max_length=8
    )

    model_inputs["labels"] = labels["input_ids"]

    return model_inputs

In [117]:
train_tokens = train_formatted.map(
    tokenize_fn,
    batched=True,
    remove_columns=train_formatted.column_names
)

eval_tokens = eval_formatted.map(
    tokenize_fn,
    batched=True,
    remove_columns=eval_formatted.column_names
)

Map:   0%|          | 0/35751 [00:00<?, ? examples/s]

Map:   0%|          | 0/8938 [00:00<?, ? examples/s]

In [118]:
# Reduce dataset size for faster CPU training

train_tokens = train_tokens.select(range(1000))
eval_tokens = eval_tokens.select(range(200))

print("Training samples:", len(train_tokens))
print("Validation samples:", len(eval_tokens))

Training samples: 1000
Validation samples: 200


In [121]:
print(train_tokens[0])

{'input_ids': [4501, 4921, 8, 826, 1506, 1108, 38, 1699, 1050, 42, 2977, 5, 3529, 10, 100, 11031, 53, 368, 29017, 22439, 1980, 2003, 11339, 148, 16, 8250, 52, 7, 41, 553, 13162, 667, 137, 29017, 22439, 16621, 4710, 19, 5250, 12, 186, 315, 14798, 7, 5, 216, 3, 7, 2033, 12817, 12, 5868, 9298, 113, 33, 8, 647, 13, 5721, 28, 112, 912, 30, 1073, 11, 2055, 18963, 5, 216, 3, 7, 92, 12817, 30, 569, 4831, 807, 114, 3, 14736, 49, 257, 13, 3850, 5452, 11, 8, 3, 102, 2242, 13, 8, 10748, 151, 6, 113, 43, 118, 365, 3, 9, 14506, 9351, 2716, 13792, 21, 4160, 5, 290, 19, 80, 119, 14798, 24, 19, 16057, 29017, 22439, 21, 2753, 10, 8, 797, 1076, 11, 887, 113, 1716, 16, 8, 2716, 5, 2703, 23752, 22439, 19, 8, 163, 4775, 113, 3, 11060, 581, 8, 7457, 615, 6, 6549, 3, 29856, 165, 9412, 3, 2375, 2420, 7, 5, 14625, 7, 13, 7457, 159, 43, 118, 4792, 6, 3, 9, 728, 3, 21176, 2717, 7908, 51, 920, 6, 11, 3, 9, 579, 10019, 47, 990, 24, 991, 12, 8, 3098, 13, 2371, 114, 27, 14408, 5, 188, 7, 21, 8, 907, 1323, 6, 20179, 7

### Observation

- The input prompts and target labels were successfully converted into numerical token IDs using the FLAN-T5 tokenizer.
- Each training example now contains three fields: input_ids, attention_mask, and labels, which are required for supervised sequence-to-sequence training.
- Padding and truncation were applied to ensure consistent input lengths across all samples.
- The original text columns were removed after tokenization because the model trains only on tokenized inputs.

In [124]:
from transformers import DataCollatorForSeq2Seq

data_collator = DataCollatorForSeq2Seq(
    tokenizer=tok,
    model=flan
)

print("Data Collator Created Successfully!")

Data Collator Created Successfully!


### Observation

* The `DataCollatorForSeq2Seq` was created successfully.
* It automatically pads input sequences and target labels to the correct length during training.
* Dynamic padding reduces unnecessary computation compared to padding every sample to a fixed maximum length.
* This collator prepares batches that are compatible with the FLAN-T5 sequence-to-sequence model.

In [129]:
from sklearn.metrics import accuracy_score
import numpy as np

def compute_metrics(eval_pred):

    predictions, labels = eval_pred

    # Decode predictions
    pred_text = tok.batch_decode(
        predictions,
        skip_special_tokens=True
    )

    # Replace ignored labels
    labels = np.where(labels != -100, labels, tok.pad_token_id)

    # Decode labels
    label_text = tok.batch_decode(
        labels,
        skip_special_tokens=True
    )

    # Convert text labels into binary labels
    pred_binary = [
        1 if p.strip().lower() == "real" else 0
        for p in pred_text
    ]

    label_binary = [
        1 if l.strip().lower() == "real" else 0
        for l in label_text
    ]

    accuracy = accuracy_score(label_binary, pred_binary)

    return {
        "accuracy": accuracy
    }

### Observation

* A custom evaluation function was defined to compute classification accuracy during validation.
* The function ignores padding tokens (represented by `-100`) so that only valid target tokens contribute to the accuracy calculation.
* Accuracy provides an intuitive measure of how often the fine-tuned model predicts the correct class label.

In [132]:
import transformers

print(transformers.__version__)

5.12.1


In [134]:
training_args = Seq2SeqTrainingArguments(

    output_dir="./flan_t5_fake_news",

    learning_rate=5e-5,

    per_device_train_batch_size=4,

    per_device_eval_batch_size=4,

    num_train_epochs=2,

    weight_decay=0.01,

    logging_steps=100,

    eval_strategy="epoch",

    save_strategy="epoch",

    predict_with_generate=True,

    report_to="none"

)

### Observation

* The training configuration specifies the learning rate, batch size, number of epochs, evaluation schedule, and model checkpoint strategy.
* A learning rate of **5 × 10⁻⁵** was selected because it is commonly used for fine-tuning transformer models.
* The model will be evaluated after each epoch, and checkpoints will be saved for future evaluation or reuse.
* Reporting to external experiment tracking tools was disabled to simplify execution within Google Colab.

In [137]:
trainer = Seq2SeqTrainer(

    model=flan,

    args=training_args,

    train_dataset=train_tokens,

    eval_dataset=eval_tokens,

    processing_class=tok,

    data_collator=data_collator,

    compute_metrics=compute_metrics

)

print("Trainer Created Successfully!")

Trainer Created Successfully!


### Observation

* The `Seq2SeqTrainer` integrates the model, tokenizer, datasets, training configuration, evaluation metrics, and data collator into a unified training pipeline.
* This abstraction simplifies the fine-tuning process while automatically handling optimization, validation, checkpointing, and metric computation.
* The training pipeline is now fully configured and ready to begin supervised fine-tuning of the FLAN-T5 model.

In [140]:
trainer.train()

Epoch,Training Loss,Validation Loss,Accuracy
1,0.482376,0.000935,1.000000
2,0.005293,0.000266,1.000000


Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

Writing model shards:   0%|          | 0/1 [00:00<?, ?it/s]

TrainOutput(global_step=500, training_loss=1.9055863312482835, metrics={'train_runtime': 19483.9971, 'train_samples_per_second': 0.103, 'train_steps_per_second': 0.026, 'total_flos': 1369514704896000.0, 'train_loss': 1.9055863312482835, 'epoch': 2.0})